### 1. Introduction and Data Loading
Credit-card fraud datasets are extremely imbalanced. In our dataset, fraud is approximately 0.38%.

In [ ]:
import json
import pandas as pd
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.data.loader import load_fraud_data
df = load_fraud_data()

### 2. Feature Engineering and Preprocessing Setup

In [ ]:
from src.preprocessing.feature_engineering import engineer_features
from src.preprocessing.pipeline import split_data, build_preprocessor
df = engineer_features(df)
X_train, X_test, y_train, y_test = split_data(df)
preprocessor = build_preprocessor(X_train)

### 3. Train Normal Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
model_baseline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight=None, solver='saga', max_iter=1000, random_state=42))
])
model_baseline.fit(X_train, y_train)

### 4. Train Class-Weighted Logistic Regression

In [ ]:
model_balanced = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', solver='saga', max_iter=1000, random_state=42))
])
model_balanced.fit(X_train, y_train)

### 5. Evaluate Both Models

In [ ]:
from src.models.evaluate import calculate_metrics
y_prob_baseline = model_baseline.predict_proba(X_test)[:, 1]
y_prob_balanced = model_balanced.predict_proba(X_test)[:, 1]
y_pred_baseline = model_baseline.predict(X_test)
y_pred_balanced = model_balanced.predict(X_test)
metrics_baseline = calculate_metrics(y_test, y_pred_baseline, y_prob_baseline)
metrics_balanced = calculate_metrics(y_test, y_pred_balanced, y_prob_balanced)

### 6. Compare Metrics

In [ ]:
import pandas as pd
metrics_df = pd.DataFrame({
    'Baseline': metrics_baseline,
    'Balanced': metrics_balanced
}).T
display(metrics_df[['precision', 'recall', 'f1', 'pr_auc', 'roc_auc']])

### 7. Display Confusion Matrices

In [ ]:
from IPython.display import Image, display
display(Image(filename='../reports/figures/phase3/phase3_logistic_confusion_matrix.png'))
display(Image(filename='../reports/figures/phase3/phase3_balanced_confusion_matrix.png'))

### 8. Display Precision-Recall Curves

In [ ]:
display(Image(filename='../reports/figures/phase3/phase3_precision_recall_curve.png'))

### 9. Display ROC Curves

In [ ]:
display(Image(filename='../reports/figures/phase3/phase3_roc_curve.png'))

### 10. Conclusion
The class-weighted model significantly improves Recall, successfully identifying a larger portion of fraudulent transactions compared to the normal baseline. However, this comes at the cost of Precision, as more legitimate transactions are falsely flagged as fraud. PR-AUC provides a solid overall summary of this trade-off.